# MIDAN — SRA Pipeline: Full Training Notebook
### ASDE Final Project — Intelligent Systems & Business Analytics
**Pipeline:** Data Prep → DBSCAN → FCM → SVM RBF → LightGBM+SHAP → SARIMA → TAS → Slack

> Run cells **top to bottom**. Each cell is one step in the pipeline.
> All models save as `.pkl` files you can load in the Streamlit dashboard.


## 0. Install Dependencies

In [ ]:
!pip install scikit-fuzzy lightgbm shap statsmodels wbgapi -q
import warnings
warnings.filterwarnings('ignore')
print("All packages installed.")


## 1. Imports & Config

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import pickle, os, json, requests
from datetime import datetime

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from sklearn.cluster import DBSCAN

import skfuzzy as fuzz
import lightgbm as lgb
import shap
from statsmodels.tsa.statespace.sarimax import SARIMAX
import wbgapi as wb

# ── CONFIG ────────────────────────────────────────────────────
plt.style.use('dark_background')
ACID      = '#C8FF57'
CARD_BG   = '#0D0D11'
TEXT_COL  = '#EDEAF8'

SECTOR_MAP = {
    'fintech':      ['Fintech', 'Financial Services', 'Payments', 'Lending', 'Insurance'],
    'ecommerce':    ['E-Commerce', 'Commerce', 'Retail', 'Marketplace'],
    'healthtech':   ['Health', 'Healthcare', 'Medical', 'Biotechnology', 'Pharma'],
    'edtech':       ['Education', 'EdTech', 'E-Learning'],
    'saas':         ['SaaS', 'Software', 'Enterprise Software', 'Cloud Computing'],
    'logistics':    ['Logistics', 'Transportation', 'Supply Chain', 'Delivery'],
    'agritech':     ['Agriculture', 'AgriTech', 'Food'],
    'other':        []
}

MODELS_DIR = 'models'
os.makedirs(MODELS_DIR, exist_ok=True)
print("Config loaded. Models will save to:", MODELS_DIR)


## 2. Upload Datasets (Run this in Colab)

In [ ]:
from google.colab import files

print("Upload the 5 CSV files:")
print("  1. big_startup_secsees_dataset.csv")
print("  2. investments_VC.csv")
print("  3. world_bank_data_2025.csv")
print("  4. unicorn_startup_companies.csv")
print("  5. all-data.csv")
print()

uploaded = files.upload()
print("\nUploaded files:", list(uploaded.keys()))


## 3. Phase 1 — Load & Clean Main Dataset

In [ ]:
# ── ISO3 → ISO2 mapping ───────────────────────────────────────
ISO3_TO_ISO2 = {
    'USA':'US','GBR':'GB','IND':'IN','CAN':'CA','AUS':'AU','DEU':'DE',
    'FRA':'FR','CHN':'CN','ISR':'IL','ESP':'ES','NLD':'NL','SWE':'SE',
    'BRA':'BR','SGP':'SG','CHE':'CH','IRL':'IE','FIN':'FI','DNK':'DK',
    'NOR':'NO','BEL':'BE','ITA':'IT','JPN':'JP','KOR':'KR','RUS':'RU',
    'ZAF':'ZA','MEX':'MX','ARG':'AR','COL':'CO','CHL':'CL','PER':'PE',
    'NGA':'NG','KEN':'KE','GHA':'GH','EGY':'EG','MAR':'MA','TUN':'TN',
    'SAU':'SA','ARE':'AE','QAT':'QA','KWT':'KW','JOR':'JO','LBN':'LB',
    'TUR':'TR','IDN':'ID','MYS':'MY','THA':'TH','PHL':'PH','VNM':'VN',
    'PAK':'PK','BGD':'BD','NZL':'NZ','HKG':'HK','TWN':'TW',
    'POL':'PL','CZE':'CZ','HUN':'HU','ROU':'RO','GRC':'GR','PRT':'PT',
    'AUT':'AT','UKR':'UA','LKA':'LK','KAZ':'KZ',
}

# ── Load big_startup dataset ──────────────────────────────────
df_raw = pd.read_csv('big_startup_secsees_dataset.csv',
                     encoding='latin-1', on_bad_lines='skip')
print(f"Raw shape: {df_raw.shape}")

# ── Convert ISO3 → ISO2 ───────────────────────────────────────
df_raw['country_code'] = df_raw['country_code'].str.upper().str.strip()
df_raw['country_iso2'] = df_raw['country_code'].map(ISO3_TO_ISO2)
df = df_raw.dropna(subset=['country_iso2']).copy()

# ── Clean numeric fields ──────────────────────────────────────
df['funding_total_usd'] = pd.to_numeric(df['funding_total_usd'], errors='coerce')
df['funding_rounds']    = pd.to_numeric(df['funding_rounds'],    errors='coerce')
df = df.dropna(subset=['funding_total_usd','funding_rounds','category_list'])
df = df[df['funding_total_usd'] > 0]
print(f"After cleaning: {df.shape}")

# ── Map sector ────────────────────────────────────────────────
def map_sector(cat_str):
    if pd.isna(cat_str): return 'other'
    cat_lower = str(cat_str).lower()
    for sector, keywords in SECTOR_MAP.items():
        if any(k.lower() in cat_lower for k in keywords):
            return sector
    return 'other'

df['sector'] = df['category_list'].apply(map_sector)

print("\nSector distribution:")
print(df['sector'].value_counts())
print("\nTop 10 countries:")
print(df['country_iso2'].value_counts().head(10))


## 4. Merge with World Bank Macro Data

In [ ]:
# ── Load World Bank data ──────────────────────────────────────
wb_raw = pd.read_csv('world_bank_data_2025.csv', encoding='utf-8', on_bad_lines='skip')
wb_raw['country_iso2'] = wb_raw['country_id'].str.upper().str.strip()
wb_raw['year']         = pd.to_numeric(wb_raw['year'], errors='coerce').astype('Int64')

# ── Use most recent available year per country (2020-2023) ───
wb_clean = wb_raw[wb_raw['year'].between(2018, 2023)].copy()
wb_clean = wb_clean[[
    'country_iso2','year',
    'Inflation (CPI %)','GDP Growth (% Annual)','Unemployment Rate (%)',
    'Public Debt (% of GDP)'
]].rename(columns={
    'Inflation (CPI %)':     'inflation',
    'GDP Growth (% Annual)': 'gdp_growth',
    'Unemployment Rate (%)': 'unemployment',
    'Public Debt (% of GDP)':'public_debt',
}).dropna(subset=['inflation','gdp_growth'])

# ── Keep LATEST year per country (most recent macro snapshot) -
wb_latest = (
    wb_clean
    .sort_values(['country_iso2','year'], ascending=[True,False])
    .drop_duplicates(subset=['country_iso2'], keep='first')
    .drop(columns=['year'])
)

print(f"World Bank latest snapshot: {wb_latest.shape}")
print(f"Countries: {wb_latest['country_iso2'].nunique()}")
print(f"\nEgypt:")
print(wb_latest[wb_latest['country_iso2']=='EG'])

# ── Merge on country only (no year dependency) ────────────────
df_merged = df.merge(wb_latest, on='country_iso2', how='inner')
print(f"\nAfter merge: {df_merged.shape}")
print(f"Countries covered: {df_merged['country_iso2'].nunique()}")
print(f"Sectors: {df_merged['sector'].value_counts().to_dict()}")


## 5. Feature Engineering — Build the 5-Feature Context Vector

In [ ]:
# ── Velocity YoY from investments_VC (founded_year) ──────────
vc = pd.read_csv('investments_VC.csv', encoding='latin-1', on_bad_lines='skip')

def map_sec(cat_str):
    if pd.isna(cat_str): return 'other'
    s = str(cat_str).lower()
    for sec, kws in SECTOR_MAP.items():
        if any(k.lower() in s for k in kws): return sec
    return 'other'

vc['cat_combined'] = vc['category_list'].fillna('') + ' ' + vc[' market '].fillna('')
vc['sector'] = vc['cat_combined'].apply(map_sec)
vc['founded_year'] = pd.to_numeric(vc['founded_year'], errors='coerce')

# Count startups founded per sector per year (2005-2014)
sy = (
    vc[vc['founded_year'].between(2005, 2014)]
    .groupby(['sector','founded_year']).size()
    .reset_index(name='founded_count')
    .sort_values(['sector','founded_year'])
)
sy['velocity_yoy'] = (
    sy.groupby('sector')['founded_count']
    .pct_change().clip(-1, 3).fillna(0)
)

# Take median velocity per sector as its "momentum" score
sector_velocity = (
    sy.groupby('sector')['velocity_yoy']
    .median().reset_index()
    .rename(columns={'velocity_yoy': 'velocity_yoy'})
)
print("Sector momentum (median YoY founding growth):")
print(sector_velocity.to_string(index=False))

# ── Aggregate: sector + country level ─────────────────────────
agg = df_merged.groupby(['sector','country_iso2']).agg(
    median_funding = ('funding_total_usd', 'median'),
    deal_count     = ('funding_total_usd', 'count'),
    inflation      = ('inflation',         'mean'),
    gdp_growth     = ('gdp_growth',        'mean'),
    unemployment   = ('unemployment',      'mean'),
).reset_index()

# ── Derived features ──────────────────────────────────────────
agg['macro_friction'] = (
    agg['inflation'] + agg['unemployment'] - agg['gdp_growth']
).clip(-50, 100)

agg['capital_concentration'] = (
    agg['median_funding'] / (agg['deal_count'] + 1)
).clip(0, 1e8)

# Merge velocity from VC
agg = agg.merge(sector_velocity, on='sector', how='left')
agg['velocity_yoy'] = agg['velocity_yoy'].fillna(0)

# ── Final 5-feature matrix ─────────────────────────────────────
FEATURES = [
    'inflation',
    'gdp_growth',
    'macro_friction',
    'capital_concentration',
    'velocity_yoy',
]

matrix = agg[['sector','country_iso2'] + FEATURES].dropna()
matrix = matrix[np.isfinite(matrix[FEATURES]).all(axis=1)]

print(f"\nFinal matrix shape: {matrix.shape}")
print(f"Sectors: {sorted(matrix['sector'].unique())}")
print(f"\nFeature stats:")
print(matrix[FEATURES].describe().round(3))
print(f"\nVelocity range: {matrix['velocity_yoy'].min():.3f} → {matrix['velocity_yoy'].max():.3f}")
print(f"Non-zero velocity rows: {(matrix['velocity_yoy'] != 0).sum()} / {len(matrix)}")

matrix.to_csv('ml_matrix_clean.csv', index=False)
print("\nSaved: ml_matrix_clean.csv")


## 6. Phase 2A — PCA Dimensionality Reduction (2D for DBSCAN + Visualization)

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

X = matrix[FEATURES].values

# ── Scale first ───────────────────────────────────────────────
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# ── PCA to 2D ─────────────────────────────────────────────────
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

print(f"PCA explained variance: {pca.explained_variance_ratio_.round(3)}")
print(f"Total variance captured: {pca.explained_variance_ratio_.sum():.1%}")

# ── Save scalers ─────────────────────────────────────────────
with open(f'{MODELS_DIR}/scaler_global.pkl',   'wb') as f: pickle.dump(scaler, f)
with open(f'{MODELS_DIR}/pca_global.pkl',      'wb') as f: pickle.dump(pca,    f)
print("Saved: scaler_global.pkl, pca_global.pkl")

# ── Plot PCA colored by sector ────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 7), facecolor=CARD_BG)
ax.set_facecolor(CARD_BG)

sectors_unique = matrix['sector'].unique()
palette = [ACID, '#5B6CF0', '#FF6B6B', '#F5A623', '#9B59B6', '#1ABC9C', '#E74C3C', '#3498DB']
color_map = {s: palette[i % len(palette)] for i, s in enumerate(sectors_unique)}

for sec in sectors_unique:
    mask = matrix['sector'].values == sec
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1],
               c=color_map[sec], label=sec, alpha=0.5, s=18, linewidths=0)

ax.set_xlabel('PCA 1', color=TEXT_COL, fontsize=12)
ax.set_ylabel('PCA 2', color=TEXT_COL, fontsize=12)
ax.set_title('PCA Projection — All Sectors', color=TEXT_COL, fontsize=14, fontweight='bold')
ax.tick_params(colors=TEXT_COL)
ax.legend(facecolor='#0D0D11', labelcolor=TEXT_COL, fontsize=9)
for spine in ax.spines.values(): spine.set_edgecolor('#2A2A3A')
plt.tight_layout()
plt.savefig(f'{MODELS_DIR}/pca_plot.png', dpi=150, facecolor=CARD_BG)
plt.show()
print("Saved: pca_plot.png")


## 7. Phase 2B — DBSCAN Clustering + FCM Soft Labels + Deterministic Naming

In [ ]:
from sklearn.cluster import DBSCAN
import skfuzzy as fuzz

# ── DBSCAN on PCA data ────────────────────────────────────────
dbscan = DBSCAN(eps=0.8, min_samples=5, n_jobs=-1)
db_labels = dbscan.fit_predict(X_pca)
n_clusters = len(set(db_labels)) - (1 if -1 in db_labels else 0)
n_noise    = int(np.sum(db_labels == -1))
print(f"DBSCAN: {n_clusters} clusters, {n_noise} noise points ({n_noise/len(db_labels):.1%})")

# ── FCM soft clustering ───────────────────────────────────────
n_fcm = max(n_clusters, 3)
cntr, u, _, _, _, _, _ = fuzz.cluster.cmeans(
    X_pca.T, c=n_fcm, m=2.0, error=0.005, maxiter=1000, init=None, seed=42
)
final_labels = np.argmax(u, axis=0)
print(f"FCM: {n_fcm} clusters | label distribution: {dict(zip(*np.unique(final_labels, return_counts=True)))}")

# ── Deterministic naming from ACTUAL cluster member medians ──
# (NOT from centroid inverse_transform — avoids PCA reconstruction artifacts)
cluster_stats = []
for i in range(n_fcm):
    mask = final_labels == i
    if mask.sum() == 0:
        continue
    data_in_cluster = X[mask]   # original unscaled values
    cluster_stats.append({
        'cluster_id':            i,
        'n_points':              int(mask.sum()),
        'inflation':             float(np.median(data_in_cluster[:, 0])),
        'gdp_growth':            float(np.median(data_in_cluster[:, 1])),
        'macro_friction':        float(np.median(data_in_cluster[:, 2])),
        'capital_concentration': float(np.median(data_in_cluster[:, 3])),
        'velocity_yoy':          float(np.median(data_in_cluster[:, 4])),
    })

cluster_df = pd.DataFrame(cluster_stats)

# Composite health score — higher = healthier market conditions
cluster_df['health_score'] = (
      cluster_df['gdp_growth']      *  2.0
    - cluster_df['inflation']       *  0.3
    + cluster_df['velocity_yoy']    *  5.0
    - cluster_df['macro_friction']  *  0.2
)

# Rank-based naming: sort by health score → assign by rank
# Guarantees at least n_fcm distinct regime labels
REGIME_NAMES = [
    'CONTRACTING_MARKET',
    'HIGH_FRICTION_MARKET',
    'EMERGING_MARKET',
    'GROWTH_MARKET',
]
cluster_df_sorted = cluster_df.sort_values('health_score').reset_index(drop=True)
cluster_names = {}
for idx, row in cluster_df_sorted.iterrows():
    name_idx = min(idx, len(REGIME_NAMES) - 1)
    cluster_names[int(row['cluster_id'])] = REGIME_NAMES[name_idx]

print("\nCluster profiles (actual data medians):")
print(cluster_df_sorted[[
    'cluster_id','n_points','inflation',
    'gdp_growth','velocity_yoy','health_score'
]].round(3).to_string(index=False))

print("\nRegime names (rank-based, deterministic):")
for k, v in cluster_names.items():
    print(f"  Cluster {k}: {v}")

# Apply names to matrix
regime_labels = np.array([cluster_names[l] for l in final_labels])
matrix['regime'] = regime_labels

print("\nFinal regime distribution:")
print(matrix['regime'].value_counts())
print(f"\nUnique regimes: {matrix['regime'].nunique()}")

import json
with open(f'{MODELS_DIR}/cluster_names.json', 'w') as f:
    json.dump(cluster_names, f, indent=2)
with open(f'{MODELS_DIR}/fcm_centers.pkl', 'wb') as f:
    pickle.dump(cntr, f)
print("\nSaved: cluster_names.json, fcm_centers.pkl")

# ── Cluster visualization ─────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 7), facecolor=CARD_BG)
ax.set_facecolor(CARD_BG)
regime_colors = {
    'GROWTH_MARKET'        : ACID,
    'HIGH_FRICTION_MARKET' : '#FF6B6B',
    'EMERGING_MARKET'      : '#5B6CF0',
    'CONTRACTING_MARKET'   : '#F5A623',
}
for regime, color in regime_colors.items():
    mask = regime_labels == regime
    if mask.sum() > 0:
        ax.scatter(X_pca[mask, 0], X_pca[mask, 1],
                   c=color, label=f'{regime} (n={mask.sum()})',
                   alpha=0.6, s=22, linewidths=0)
ax.set_xlabel('PCA 1', color=TEXT_COL); ax.set_ylabel('PCA 2', color=TEXT_COL)
ax.set_title('Market Regime Clusters — DBSCAN + FCM', color=TEXT_COL, fontsize=13, fontweight='bold')
ax.tick_params(colors=TEXT_COL)
ax.legend(facecolor='#0D0D11', labelcolor=TEXT_COL, fontsize=9)
for sp in ax.spines.values(): sp.set_edgecolor('#2A2A3A')
plt.tight_layout()
plt.savefig(f'{MODELS_DIR}/dbscan_clusters.png', dpi=150, facecolor=CARD_BG)
plt.show()
print("Saved: dbscan_clusters.png")


## 8. Phase 2C — SVM RBF Training (Global + Per-Sector)

In [ ]:
from sklearn.svm import SVC
from sklearn.metrics import classification_report

le = LabelEncoder()
y  = le.fit_transform(matrix['regime'])

with open(f'{MODELS_DIR}/label_encoder.pkl', 'wb') as f:
    pickle.dump(le, f)
print(f"Classes: {list(le.classes_)}")

# ── GLOBAL MODEL ──────────────────────────────────────────────
X_scaled_full = scaler.transform(matrix[FEATURES].values)
X_tr, X_te, y_tr, y_te = train_test_split(X_scaled_full, y,
                                           test_size=0.2, random_state=42,
                                           stratify=y)

svm_global = SVC(kernel='rbf', C=5.0, gamma='scale', probability=True, random_state=42)
svm_global.fit(X_tr, y_tr)

y_pred = svm_global.predict(X_te)
acc    = accuracy_score(y_te, y_pred)
print(f"\nGlobal SVM Accuracy: {acc:.3f}")
print(classification_report(y_te, y_pred, target_names=le.classes_))

with open(f'{MODELS_DIR}/svm_global.pkl', 'wb') as f:
    pickle.dump(svm_global, f)
print("Saved: svm_global.pkl")

# ── SECTOR-SPECIFIC MODELS (>= 500 rows) ─────────────────────
sector_counts = matrix['sector'].value_counts()
large_sectors = sector_counts[sector_counts >= 500].index.tolist()
print(f"\nBuilding sector-specific models for: {large_sectors}")

sector_models = {}
for sec in large_sectors:
    mask   = matrix['sector'] == sec
    X_sec  = scaler.transform(matrix.loc[mask, FEATURES].values)
    y_sec  = y[mask]

    if len(np.unique(y_sec)) < 2:
        print(f"  {sec}: only 1 class, skipping")
        continue

    X_str, X_ste, y_str, y_ste = train_test_split(X_sec, y_sec,
                                                   test_size=0.2,
                                                   random_state=42)
    svm_sec = SVC(kernel='rbf', C=5.0, gamma='scale', probability=True, random_state=42)
    svm_sec.fit(X_str, y_str)
    sec_acc = accuracy_score(y_ste, svm_sec.predict(X_ste))

    fname = f'{MODELS_DIR}/svm_{sec}.pkl'
    with open(fname, 'wb') as f:
        pickle.dump(svm_sec, f)

    sector_models[sec] = fname
    print(f"  {sec}: {mask.sum()} rows | Accuracy: {sec_acc:.3f} | Saved: {fname}")

with open(f'{MODELS_DIR}/sector_model_index.json', 'w') as f:
    json.dump({'large_sectors': large_sectors, 'threshold': 500}, f, indent=2)
print("\nSaved: sector_model_index.json")


## 9. Phase 3A — LightGBM Surrogate + SHAP Explainability

In [ ]:
import lightgbm as lgb
import shap

# ── Train LightGBM to mimic SVM ───────────────────────────────
# Use SVM global predictions as "ground truth" labels for surrogate
svm_preds_full = svm_global.predict(X_scaled_full)

lgb_model = lgb.LGBMClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42,
    verbose=-1,
)
lgb_model.fit(X_scaled_full, svm_preds_full)

surrogate_acc = accuracy_score(svm_preds_full, lgb_model.predict(X_scaled_full))
print(f"LightGBM surrogate fidelity (mirrors SVM): {surrogate_acc:.3f}")
print("(>= 0.95 is excellent, means SHAP explanations are valid)")

with open(f'{MODELS_DIR}/lgb_surrogate.pkl', 'wb') as f:
    pickle.dump(lgb_model, f)
print("Saved: lgb_surrogate.pkl")

# ── Compute SHAP values on sample ────────────────────────────
sample_idx = np.random.choice(len(X_scaled_full), min(500, len(X_scaled_full)), replace=False)
X_sample   = X_scaled_full[sample_idx]

explainer   = shap.TreeExplainer(lgb_model)
shap_values = explainer.shap_values(X_sample)

# For multiclass, take mean abs across all classes
if isinstance(shap_values, list):
    mean_shap = np.mean([np.abs(sv) for sv in shap_values], axis=0)
else:
    mean_shap = np.abs(shap_values)

feature_importance = dict(zip(FEATURES, mean_shap.mean(axis=0)))
fi_sorted = dict(sorted(feature_importance.items(), key=lambda x: x[1], reverse=True))

print("\nGlobal SHAP Feature Importance:")
for feat, val in fi_sorted.items():
    bar = '█' * int(val * 30)
    print(f"  {feat:<30} {bar} {val:.4f}")

with open(f'{MODELS_DIR}/shap_feature_importance.json', 'w') as f:
    json.dump(fi_sorted, f, indent=2)

# ── SHAP Summary Plot ─────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5), facecolor=CARD_BG)
ax.set_facecolor(CARD_BG)

feats = list(fi_sorted.keys())
vals  = list(fi_sorted.values())
colors = [ACID if v == max(vals) else '#5B6CF0' if v > np.median(vals) else '#4A4A5E' for v in vals]

bars = ax.barh(feats, vals, color=colors, height=0.55)
for bar, val in zip(bars, vals):
    ax.text(val + 0.001, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', color=TEXT_COL, fontsize=10)

ax.set_xlabel('Mean |SHAP Value|', color=TEXT_COL)
ax.set_title('SHAP Feature Importance — Global Model', color=TEXT_COL, fontsize=13, fontweight='bold')
ax.tick_params(colors=TEXT_COL)
ax.invert_yaxis()
for spine in ax.spines.values(): spine.set_edgecolor('#2A2A3A')
plt.tight_layout()
plt.savefig(f'{MODELS_DIR}/shap_importance.png', dpi=150, facecolor=CARD_BG)
plt.show()
print("Saved: shap_importance.png")


## 10. Phase 3B — SARIMA Time-Series Forecast + Drift Detection

In [ ]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

# ── Load investments_VC.csv ───────────────────────────────────
vc = pd.read_csv('investments_VC.csv', encoding='latin-1', on_bad_lines='skip')
print(f"VC dataset: {vc.shape}")

# ── Build monthly time series ─────────────────────────────────
vc['last_funding_at'] = pd.to_datetime(vc['last_funding_at'], errors='coerce')
vc = vc.dropna(subset=['last_funding_at'])
vc['month'] = vc['last_funding_at'].dt.to_period('M')

# Apply sector mapping
vc['category_list'] = vc['category_list'].fillna('') + ' ' + vc[' market '].fillna('')

def map_sector_vc(cat_str):
    if pd.isna(cat_str): return 'other'
    cat_lower = str(cat_str).lower()
    for sector, keywords in SECTOR_MAP.items():
        if any(k.lower() in cat_lower for k in keywords):
            return sector
    return 'other'

vc['sector'] = vc['category_list'].apply(map_sector_vc)

sarima_models = {}

for sec in ['fintech', 'ecommerce', 'healthtech', 'edtech', 'saas']:
    ts = (vc[vc['sector'] == sec]
          .groupby('month')
          .size()
          .reset_index(name='deals'))

    if len(ts) < 24:
        print(f"  {sec}: insufficient data ({len(ts)} months), skipping")
        continue

    ts = ts.set_index('month').sort_index()
    ts.index = ts.index.to_timestamp()

    try:
        model = SARIMAX(ts['deals'],
                        order=(1, 1, 1),
                        seasonal_order=(1, 1, 1, 12),
                        enforce_stationarity=False,
                        enforce_invertibility=False)
        result = model.fit(disp=False)

        # ── Drift Detection ─────────────────────────────────────
        residuals = result.resid
        std_resid  = residuals.std()
        drift_flag = bool((np.abs(residuals) > 3 * std_resid).sum() > 3)

        # ── Forecast 90 days (~3 months) ────────────────────────
        forecast     = result.get_forecast(steps=3)
        forecast_df  = forecast.summary_frame(alpha=0.1)

        sarima_models[sec] = {
            'aic':          round(result.aic, 2),
            'drift_flag':   drift_flag,
            'last_date':    str(ts.index[-1].date()),
            'forecast_mean': forecast_df['mean'].tolist(),
            'forecast_lower': forecast_df['mean_ci_lower'].tolist(),
            'forecast_upper': forecast_df['mean_ci_upper'].tolist(),
        }

        # ── Save full result ─────────────────────────────────────
        with open(f'{MODELS_DIR}/sarima_{sec}.pkl', 'wb') as f:
            pickle.dump(result, f)

        print(f"  {sec}: AIC={result.aic:.1f} | Drift={'YES ⚠' if drift_flag else 'No'} | Saved")

        # ── Plot ─────────────────────────────────────────────────
        fig, ax = plt.subplots(figsize=(12, 4), facecolor=CARD_BG)
        ax.set_facecolor(CARD_BG)
        ax.plot(ts.index, ts['deals'], color='#5B6CF0', linewidth=2, label='Historical')
        fc_dates = pd.date_range(ts.index[-1], periods=4, freq='MS')[1:]
        ax.plot(fc_dates, forecast_df['mean'], color=ACID, linewidth=2.5, linestyle='--', label='Forecast')
        ax.fill_between(fc_dates,
                        forecast_df['mean_ci_lower'],
                        forecast_df['mean_ci_upper'],
                        alpha=0.15, color=ACID)
        ax.axvline(ts.index[-1], color='#4A4A5E', linestyle=':', linewidth=1)
        if drift_flag:
            ax.text(ts.index[int(len(ts)*0.05)], ts['deals'].max() * 0.9,
                    'DRIFT DETECTED', color='#FF6B6B', fontsize=9, fontweight='bold')
        ax.set_title(f'SARIMA Forecast — {sec.title()}', color=TEXT_COL, fontsize=12, fontweight='bold')
        ax.set_xlabel('Date', color=TEXT_COL)
        ax.set_ylabel('Deal Count', color=TEXT_COL)
        ax.tick_params(colors=TEXT_COL)
        ax.legend(facecolor='#0D0D11', labelcolor=TEXT_COL, fontsize=9)
        for spine in ax.spines.values(): spine.set_edgecolor('#2A2A3A')
        plt.tight_layout()
        plt.savefig(f'{MODELS_DIR}/sarima_{sec}.png', dpi=130, facecolor=CARD_BG)
        plt.show()

    except Exception as e:
        print(f"  {sec}: SARIMA failed — {e}")

with open(f'{MODELS_DIR}/sarima_results.json', 'w') as f:
    json.dump(sarima_models, f, indent=2)
print("\nSaved: sarima_results.json")


## 11. Phase 1 (Live) — Inference Bridge: Build X_new from User Idea

In [ ]:
import wbgapi as wb_api

def get_live_macro(country_iso2: str) -> dict:
    """Fetch live macro from World Bank API."""
    iso2 = country_iso2.upper()
    defaults = {'inflation': 10.0, 'gdp_growth': 3.0, 'unemployment': 7.0}
    indicators = {
        'FP.CPI.TOTL.ZG':    'inflation',
        'NY.GDP.MKTP.KD.ZG': 'gdp_growth',
        'SL.UEM.TOTL.ZS':    'unemployment',
    }
    result = dict(defaults)
    for code, name in indicators.items():
        try:
            data = wb_api.data.get(code, iso2, mrv=1)
            val  = list(data.values())[0] if data else None
            if val is not None:
                result[name] = round(float(val), 2)
        except:
            pass
    return result

def get_sector_velocity(sector: str, matrix_df: pd.DataFrame) -> float:
    """Get sector velocity from training matrix."""
    sub = matrix_df[matrix_df['sector'] == sector.lower()]
    if len(sub) == 0:
        return 0.0
    return float(sub['velocity_yoy'].median())

def build_x_new(sector: str, country_iso2: str, matrix_df: pd.DataFrame,
                scaler_obj, features: list) -> np.ndarray:
    """Build inference context vector X_new."""
    print(f"Building X_new for: {sector} / {country_iso2.upper()}")

    macro    = get_live_macro(country_iso2)
    velocity = get_sector_velocity(sector, matrix_df)
    print(f"  Live macro: {macro}")
    print(f"  Sector velocity: {velocity:.3f}")

    inflation      = macro['inflation']
    gdp_growth     = macro['gdp_growth']
    unemployment   = macro['unemployment']
    macro_friction = inflation + unemployment - gdp_growth

    # Capital concentration: use sector median from training data
    sec_mask   = matrix_df[matrix_df['sector'] == sector.lower()]
    cap_conc   = float(sec_mask['capital_concentration'].median()) if len(sec_mask) > 0 else 500000.0

    x_raw = np.array([[
        inflation,
        gdp_growth,
        macro_friction,
        cap_conc,
        velocity,
    ]])

    x_scaled = scaler_obj.transform(x_raw)
    print(f"  X_new (scaled): {x_scaled.round(3)}")
    return x_scaled

# ── TEST ──────────────────────────────────────────────────────
X_new_test = build_x_new('fintech', 'EG', matrix, scaler, FEATURES)
print("\nX_new built successfully.")


## 12. Phase 2-5 (Live) — Full Inference Pipeline

In [ ]:
def run_full_inference(sector: str, country: str):
    """
    Full SRA pipeline inference.
    Returns: regime, confidence, shap_top3, sarima_outlook, TAS, report
    """
    print(f"\n{'='*55}")
    print(f"  MIDAN PIPELINE — {sector.upper()} / {country.upper()}")
    print(f"{'='*55}")

    # ── 1. Build X_new ────────────────────────────────────────
    X_new = build_x_new(sector, country, matrix, scaler, FEATURES)

    # ── 2. Router: sector-specific or global model ────────────
    sector_model_path = f'{MODELS_DIR}/svm_{sector.lower()}.pkl'
    if os.path.exists(sector_model_path):
        with open(sector_model_path, 'rb') as f:
            svm_model = pickle.load(f)
        print(f"\n[STEP3] Using sector-specific SVM: {sector}")
    else:
        svm_model = svm_global
        print(f"\n[STEP3] Using global SVM (fallback)")

    # ── 3. SVM Classification ─────────────────────────────────
    pred_encoded = svm_model.predict(X_new)[0]
    proba        = svm_model.predict_proba(X_new)[0]
    confidence   = round(float(proba.max()), 3)

    with open(f'{MODELS_DIR}/label_encoder.pkl', 'rb') as f:
        le_loaded = pickle.load(f)
    regime = le_loaded.inverse_transform([pred_encoded])[0]

    print(f"[STEP3] Regime: {regime} | Confidence: {confidence:.1%}")

    # ── 4A. SHAP Explanation ──────────────────────────────────
    with open(f'{MODELS_DIR}/lgb_surrogate.pkl', 'rb') as f:
        lgb_loaded = pickle.load(f)

    explainer_live = shap.TreeExplainer(lgb_loaded)
    shap_live      = explainer_live.shap_values(X_new)

    if isinstance(shap_live, list):
        shap_arr = np.mean([np.abs(sv) for sv in shap_live], axis=0)[0]
    else:
        shap_arr = np.abs(shap_live)[0]

    shap_dict = dict(zip(FEATURES, shap_arr))
    shap_sorted = sorted(shap_dict.items(), key=lambda x: x[1], reverse=True)
    xai_score = float(confidence * np.mean(shap_arr))
    print(f"[STEP4A] Top signal: {shap_sorted[0][0]} ({shap_sorted[0][1]:.3f})")

    # ── 4B. SARIMA Outlook ────────────────────────────────────
    sarima_trend = 0.5  # default neutral
    with open(f'{MODELS_DIR}/sarima_results.json') as f:
        sarima_data = json.load(f)

    sec_key = sector.lower()
    if sec_key in sarima_data:
        fc = sarima_data[sec_key]['forecast_mean']
        drift = sarima_data[sec_key]['drift_flag']
        sarima_trend = 0.75 if fc[-1] > fc[0] else 0.35
        if drift:
            print(f"[STEP4B] DRIFT DETECTED — Manual Reclustering Advised")
        print(f"[STEP4B] SARIMA trend score: {sarima_trend}")

    # ── 5. TAS Score ──────────────────────────────────────────
    tas = round(confidence * 0.40 + sarima_trend * 0.35 + xai_score * 0.25, 3)
    print(f"\n[STEP5] TAS = {confidence:.2f}×0.40 + {sarima_trend:.2f}×0.35 + {xai_score:.2f}×0.25 = {tas:.3f}")

    # ── Slack Action ──────────────────────────────────────────
    if tas >= 0.70 and 'GROWTH' in regime:
        print(f"[SLACK] TAS >= 0.70 & GROWTH_MARKET → Webhook would fire")
    else:
        print(f"[SLACK] TAS={tas} — No action triggered")

    # ── Trinity Report ────────────────────────────────────────
    top3 = [f[0] for f in shap_sorted[:3]]
    report = {
        'finding':     f"Market classified as {regime} with {confidence:.0%} confidence. "
                       f"Top signals: {', '.join(top3)}.",
        'implication': f"{'High confidence in growth conditions.' if tas >= 0.7 else 'Market shows friction — proceed with caution.'} "
                       f"SARIMA 90-day outlook: {'positive' if sarima_trend > 0.5 else 'neutral/declining'}.",
        'action':      f"{'MOVE NOW — apply to Flat6Labs/Cairo Angels within 30 days.' if tas >= 0.7 else 'Validate demand first. Run 20 customer interviews before building.'} "
                       f"Key signal to watch: {top3[0].replace('_', ' ')}.",
        'tas':         tas,
        'regime':      regime,
        'confidence':  confidence,
    }

    print(f"\n{'─'*55}")
    print("TRINITY REPORT")
    print(f"{'─'*55}")
    for k, v in report.items():
        if k in ['finding', 'implication', 'action']:
            print(f"  {k.upper()}: {v}")
    print(f"{'─'*55}\n")

    return report

# ── RUN A TEST ────────────────────────────────────────────────
result = run_full_inference('fintech', 'EG')


## 13. Agent A1 — NLP Parser (LLM with Temperature=0)

In [ ]:
# ═══════════════════════════════════════════════════════════════
# AGENT A1 — NLP Parser (Groq LLM, Temperature=0)
# Free tier: 14,400 requests/day — console.groq.com
# ═══════════════════════════════════════════════════════════════
!pip install groq -q

import os, json, re
from groq import Groq

# ── Constants ─────────────────────────────────────────────────
VALID_SECTORS = [
    'fintech', 'ecommerce', 'healthtech', 'edtech',
    'saas', 'logistics', 'agritech', 'other'
]
VALID_COUNTRIES = [
    'EG','SA','AE','US','GB','DE','FR','IN','PK',
    'NG','KE','MA','TN','JO','LB','KW','QA','TR','ID','MY'
]

KEYWORD_FALLBACK_SECTORS = {
    'fintech':    ['finance','payment','fintech','bank','loan',
                   'lending','invoice','insurance','wallet','money'],
    'ecommerce':  ['ecommerce','e-commerce','shop','store',
                   'retail','marketplace','delivery','commerce'],
    'healthtech': ['health','medical','doctor','clinic',
                   'hospital','pharma','biotech','mental'],
    'edtech':     ['education','learning','school','university',
                   'course','tutor','edtech','training'],
    'saas':       ['saas','software','platform','dashboard',
                   'tool','api','enterprise','cloud','b2b','crm'],
    'logistics':  ['logistics','shipping','supply chain',
                   'warehouse','transport','fleet','trucking'],
    'agritech':   ['agri','farm','crop','harvest',
                   'food','agriculture','irrigation'],
}
KEYWORD_FALLBACK_COUNTRIES = {
    'EG': ['egypt','cairo','egyptian','مصر','القاهرة'],
    'SA': ['saudi','ksa','riyadh','jeddah','السعودية'],
    'AE': ['uae','dubai','abu dhabi','emirates','الإمارات'],
    'MA': ['morocco','casablanca','المغرب'],
    'NG': ['nigeria','lagos','abuja'],
    'KE': ['kenya','nairobi'],
    'TN': ['tunisia','tunis','تونس'],
    'JO': ['jordan','amman','الأردن'],
}

# ── Helper: parse + validate LLM response ─────────────────────
def parse_llm_response(raw: str) -> dict | None:
    try:
        data = json.loads(raw.strip())
    except json.JSONDecodeError:
        match = re.search(r'\{[^{}]+\}', raw, re.DOTALL)
        if not match: return None
        try: data = json.loads(match.group())
        except: return None

    sector  = str(data.get('sector',  '')).lower().strip()
    country = str(data.get('country', '')).upper().strip()
    if sector  not in VALID_SECTORS:   sector  = 'other'
    if country not in VALID_COUNTRIES: country = 'EG'
    return {'sector': sector, 'country': country}

# ── Helper: keyword fallback (no API needed) ──────────────────
def keyword_fallback(idea: str) -> dict:
    text = idea.lower()
    sector = 'other'
    for sec, kws in KEYWORD_FALLBACK_SECTORS.items():
        if any(k in text for k in kws):
            sector = sec
            break
    country = 'EG'
    for code, kws in KEYWORD_FALLBACK_COUNTRIES.items():
        if any(k in text for k in kws):
            country = code
            break
    return {'sector': sector, 'country': country}

# ── Main Agent A1 function ────────────────────────────────────
def agent_a1_parse(idea: str, groq_client=None) -> dict:
    """
    Parse startup idea into {sector, country}.
    Uses Groq LLM (Temperature=0) if client provided,
    falls back to keyword matching automatically.
    """
    SYSTEM_PROMPT = """You are a startup idea parser. Extract ONLY the sector and country.
Reply with EXACTLY this JSON and nothing else:
{"sector": "...", "country": "ISO2_CODE"}

Valid sectors: fintech, ecommerce, healthtech, edtech, saas, logistics, agritech, other
Valid country codes: EG, SA, AE, US, GB, DE, FR, IN, PK, NG, KE, MA, TN, JO, LB, KW, QA, TR
Default country = EG if not mentioned in the idea."""

    if groq_client is not None:
        try:
            response = groq_client.chat.completions.create(
                model="llama-3.1-8b-instant",
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user",   "content": idea},
                ],
                temperature=0.0,
                max_tokens=60,
                response_format={"type": "json_object"},
            )
            raw    = response.choices[0].message.content
            parsed = parse_llm_response(raw)
            if parsed:
                print(f"[A1] LLM result: {parsed}")
                return parsed
            print(f"[A1] LLM returned unparseable output, using fallback")
        except Exception as e:
            print(f"[A1] LLM error ({e}), using keyword fallback")

    result = keyword_fallback(idea)
    print(f"[A1] Keyword fallback result: {result}")
    return result

# ── Setup Groq client ─────────────────────────────────────────
# Paste your Groq key here (get free key from console.groq.com)
GROQ_API_KEY = "your_groq_key_here"

if GROQ_API_KEY != "your_groq_key_here":
    groq_client = Groq(api_key=GROQ_API_KEY)
    print("Groq client ready.")
else:
    groq_client = None
    print("No API key set — using keyword fallback mode.")

# ── TEST ──────────────────────────────────────────────────────
test_ideas = [
    "Invoice financing app for Egyptian SMEs",
    "E-commerce last-mile delivery platform in Cairo",
    "Mental health app for Saudi Arabia",
    "SaaS CRM for logistics companies in UAE",
    "Agricultural irrigation system for Morocco farmers",
]

print("\n" + "="*55)
print("Agent A1 — Startup Idea Parser")
print("="*55)
for idea in test_ideas:
    result = agent_a1_parse(idea, groq_client)
    print(f"Idea:   {idea}")
    print(f"Output: sector={result['sector']} | country={result['country']}")
    print()


## 14. Build Context Files for Agents A2 & A4

In [ ]:
# ── Unicorn Competitors (Agent A2) ────────────────────────────
unicorns = pd.read_csv('unicorn_startup_companies.csv', encoding='utf-8', on_bad_lines='skip')
print(f"Unicorns: {unicorns.shape}")
print(unicorns.head(3))

def map_unicorn_sector(industry_str):
    if pd.isna(industry_str): return 'other'
    text = str(industry_str).lower()
    for sector, keywords in SECTOR_MAP.items():
        if any(k.lower() in text for k in keywords):
            return sector
    return 'other'

unicorns['sector'] = unicorns['Industry'].apply(map_unicorn_sector)

competitors_by_sector = {}
for sec in unicorns['sector'].unique():
    sub = unicorns[unicorns['sector'] == sec][['Company', 'Valuation ($B)', 'Country', 'Industry']].head(10)
    competitors_by_sector[sec] = sub.to_dict(orient='records')

with open(f'{MODELS_DIR}/competitors_context.json', 'w') as f:
    json.dump(competitors_by_sector, f, indent=2, default=str)

print("\nCompetitors by sector:")
for sec, comps in competitors_by_sector.items():
    print(f"  {sec}: {len(comps)} unicorn competitors")
print("Saved: competitors_context.json")

# ── Sentiment / Demand Data (Agent A4) ────────────────────────
try:
    sentiment = pd.read_csv('all-data.csv', encoding='latin-1',
                            names=['sentiment', 'text'], on_bad_lines='skip')
    print(f"\nSentiment data: {sentiment.shape}")
    print(sentiment['sentiment'].value_counts())

    # Save as context
    sentiment_sample = sentiment.sample(min(200, len(sentiment)), random_state=42)
    with open(f'{MODELS_DIR}/sentiment_context.json', 'w') as f:
        json.dump(sentiment_sample.to_dict(orient='records'), f, indent=2, default=str)
    print("Saved: sentiment_context.json")
except Exception as e:
    print(f"Sentiment data issue: {e}")


## 15. Download All Models (for Streamlit Dashboard)

In [ ]:
import zipfile
from google.colab import files

# ── List all saved models ─────────────────────────────────────
saved_files = os.listdir(MODELS_DIR)
print("Files in models/ directory:")
for f in sorted(saved_files):
    size = os.path.getsize(f'{MODELS_DIR}/{f}')
    print(f"  {f:<40} {size/1024:.1f} KB")

# ── Zip everything ────────────────────────────────────────────
zip_path = 'midan_models.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for f in saved_files:
        zf.write(f'{MODELS_DIR}/{f}', f)

zip_size = os.path.getsize(zip_path) / (1024*1024)
print(f"\nZipped all models → {zip_path} ({zip_size:.1f} MB)")

# ── Download ──────────────────────────────────────────────────
files.download(zip_path)
print("Download started!")


## Pipeline Complete

### Models Saved
| File | Purpose |
|---|---|
| `svm_global.pkl` | Global SVM classifier (fallback) |
| `svm_fintech.pkl` etc | Sector-specific SVMs |
| `lgb_surrogate.pkl` | LightGBM mirrors SVM for SHAP |
| `scaler_global.pkl` | StandardScaler for X_new |
| `pca_global.pkl` | PCA 2D for visualization |
| `label_encoder.pkl` | Regime name ↔ integer mapping |
| `sarima_*.pkl` | SARIMA per sector |
| `cluster_names.json` | Deterministic cluster labels |
| `sarima_results.json` | 90-day forecasts + drift flags |
| `competitors_context.json` | Agent A2 competitor data |
| `sentiment_context.json` | Agent A4 demand signals |

### Next Step
Place `midan_models.zip` → extract into `models/` folder in your Streamlit project.
The `app.py` dashboard will auto-load all `.pkl` files.
